# Cohort Selection and Table 1 Audit
**Manuscript:** *Automated segmentation of lung nodules in CT scans using a U-Net deep learning model: A study based on the LIDC-IDRI dataset*

This reproducibility notebook audits the scan- and patient-level denominators used in cohort selection and Table 1. It uses only frozen project metadata and does **not** rerun preprocessing, cohort selection, model training, or segmentation evaluation.

It reconstructs the eligible population, selected cohort, patient-disjoint comparator, origin of the previous 512-patient denominator, and corrected DICOM-file comparison.


In [1]:
from pathlib import Path
import pandas as pd
from scipy.stats import mannwhitneyu
from IPython.display import display

cwd=Path.cwd().resolve()
if (cwd/"metadata").exists(): PROJECT_ROOT=cwd
elif (cwd.parent/"metadata").exists(): PROJECT_ROOT=cwd.parent
else: raise FileNotFoundError("Set PROJECT_ROOT manually to the repository root.")
M=PROJECT_ROOT/"metadata"
A=M/"lidc_annotation_inventory.csv"
C=M/"selected_500_scan_cohort_seed42.csv"
D=M/"lidc_download_inventory.csv"
for p in [A,C,D]:
    if not p.exists(): raise FileNotFoundError(p)
print("PROJECT_ROOT:",PROJECT_ROOT)


PROJECT_ROOT: /Volumes/KINGSTON/PLOS_2D_UNet_Revision


## Load frozen metadata
The annotation inventory is scan-level; the download inventory is patient-directory-level and supplies `dcm_files_with_extension`.


In [2]:
ann=pd.read_csv(A); cohort=pd.read_csv(C); download=pd.read_csv(D)
for df in [ann,cohort,download]: df["patient_id"]=df["patient_id"].astype(str)
ann["scan_id"]=ann["scan_id"].astype(int); cohort["scan_id"]=cohort["scan_id"].astype(int)
print("Annotation inventory:",ann.shape,"patients:",ann.patient_id.nunique())
print("Selected cohort:",cohort.shape,"patients:",cohort.patient_id.nunique())
print("Download inventory:",download.shape,"patients:",download.patient_id.nunique())


Annotation inventory: (1018, 7) patients: 1010
Selected cohort: (500, 7) patients: 498
Download inventory: (1010, 3) patients: 1010


## Reconstruct the eligible population and patient-disjoint comparator
The corrected representativeness comparison is restricted to the eligible annotated sampling frame. Residual scans from patients already represented in the selected cohort are excluded from the comparator so that the groups are patient-disjoint.


In [3]:
eligible=ann[(ann.status=="OK") & (ann.has_nodule_annotations==True)].copy()
negative=ann[(ann.status=="OK") & (ann.has_nodule_annotations==False)].copy()
selected_scans=set(cohort.scan_id); selected_patients=set(cohort.patient_id)
remaining=eligible[~eligible.scan_id.isin(selected_scans)].copy()
remaining_patients=set(remaining.patient_id)
overlap=selected_patients & remaining_patients
comparator=remaining[~remaining.patient_id.isin(selected_patients)].copy()
comparator_patients=set(comparator.patient_id)
excluded=remaining[remaining.patient_id.isin(selected_patients)].copy()

summary=pd.DataFrame([
 ["Full annotation inventory",len(ann),ann.patient_id.nunique()],
 ["Eligible annotated population",len(eligible),eligible.patient_id.nunique()],
 ["Selected cohort",len(cohort),len(selected_patients)],
 ["All remaining eligible",len(remaining),len(remaining_patients)],
 ["Residual scans belonging to selected patients",len(excluded),excluded.patient_id.nunique()],
 ["Patient-disjoint eligible comparator",len(comparator),len(comparator_patients)],
 ["Annotation-negative population",len(negative),negative.patient_id.nunique()],
],columns=["Population","Scans","Patients"])
display(summary)
print("Overlapping patient IDs:",sorted(overlap))
print(f"Eligible decomposition: {len(eligible)} = {len(cohort)} + {len(comparator)} + {len(excluded)}")


,Population,Scans,Patients
0,Full annotation inventory,1018,1010
1,Eligible annotated population,883,875
2,Selected cohort,500,498
3,All remaining eligible,383,382
4,Residual scans belonging to selected patients,5,5
5,Patient-disjoint eligible comparator,378,377
6,Annotation-negative population,135,135


Overlapping patient IDs: ['LIDC-IDRI-0132', 'LIDC-IDRI-0151', 'LIDC-IDRI-0315', 'LIDC-IDRI-0355', 'LIDC-IDRI-0484']
Eligible decomposition: 883 = 500 + 378 + 5


## Verify scan multiplicity and annotation-negative population


In [4]:
counts=eligible.groupby("patient_id").size()
display(counts.value_counts().sort_index().rename_axis("eligible_scans_per_patient").reset_index(name="patients"))
negative_patients=set(negative.patient_id)
negative_overlap=negative_patients & set(eligible.patient_id)
print("Annotation-negative patients:",len(negative_patients))
print("Overlap with eligible patients:",len(negative_overlap))


,eligible_scans_per_patient,patients
0,1,867
1,2,8


Annotation-negative patients: 135
Overlap with eligible patients: 0


DICOM-file counts are summarized by the median and interquartile range (IQR). For presentation in Table 1, these descriptive statistics are reported to the nearest whole DICOM file; inferential testing uses the unrounded patient-level values.

## Corrected patient-level DICOM comparison
The corrected Table 1 comparison uses the 498 selected patients and 377 patients in the patient-disjoint eligible comparator.


In [5]:
dcm=(download[["patient_id","dcm_files_with_extension"]]
     .drop_duplicates("patient_id").set_index("patient_id")["dcm_files_with_extension"])
sel=dcm.reindex(sorted(selected_patients)).dropna()
cmp=dcm.reindex(sorted(comparator_patients)).dropna()

def stats(x):
    return len(x),float(x.median()),float(x.quantile(.25)),float(x.quantile(.75))
ns,ms,q1s,q3s=stats(sel); nc,mc,q1c,q3c=stats(cmp)
U,p=mannwhitneyu(sel,cmp,alternative="two-sided")
display(pd.DataFrame([
 ["Selected cohort",ns,ms,q1s,q3s],
 ["Patient-disjoint eligible comparator",nc,mc,q1c,q3c]
],columns=["Group","Patients","Median DICOM files","Q1","Q3"]))
print(f"Mann-Whitney U = {U:.1f}")
print(f"Two-sided p = {p:.6f}")


,Group,Patients,Median DICOM files,Q1,Q3
0,Selected cohort,498,443.0,270.0,579.5
1,Patient-disjoint eligible comparator,377,470.0,270.0,604.0


Mann-Whitney U = 89461.0
Two-sided p = 0.233373


## Provenance of the previous 512-patient denominator
The complete download inventory contains 1,010 unique patient directories. Removing the 498 selected patients gives 512 other patient directories. This broader group extends beyond the eligible annotated sampling frame and is therefore not used in the corrected Table 1.


In [6]:
all_patients=set(download.patient_id)
old_patients=all_patients-selected_patients
old=dcm.reindex(sorted(old_patients)).dropna()
no,mo,q1o,q3o=stats(old)
Uold,pold=mannwhitneyu(sel,old,alternative="two-sided")
print("Download-inventory patients:",len(all_patients))
print("Selected patients:",len(selected_patients))
print("Other inventory patients:",len(old_patients))
print(f"Old comparison: {ms:.0f} ({q1s:.0f}-{q3s:.0f}) vs {mo:.0f} ({q1o:.0f}-{q3o:.0f})")
print(f"Old Mann-Whitney U = {Uold:.1f}; p = {pold:.6f}")


Download-inventory patients: 1010
Selected patients: 498
Other inventory patients: 512
Old comparison: 443 (270-580) vs 409 (266-580)
Old Mann-Whitney U = 129836.5; p = 0.612411


In [7]:
print("SELECTED RAW:")
print("median =", repr(ms))
print("Q1     =", repr(q1s))
print("Q3     =", repr(q3s))

print("\nCOMPARATOR RAW:")
print("median =", repr(mc))
print("Q1     =", repr(q1c))
print("Q3     =", repr(q3c))

print("\nTEST:")
print("U =", repr(U))
print("p =", repr(p))

print("\nROUNDED FOR MANUSCRIPT:")
print(
    f"Selected   : {ms:.0f} "
    f"({q1s:.0f}–{q3s:.0f})"
)
print(
    f"Comparator : {mc:.0f} "
    f"({q1c:.0f}–{q3c:.0f})"
)

SELECTED RAW:
median = 443.0
Q1     = 270.0
Q3     = 579.5

COMPARATOR RAW:
median = 470.0
Q1     = 270.0
Q3     = 604.0

TEST:
U = 89461.0
p = 0.23337301385483478

ROUNDED FOR MANUSCRIPT:
Selected   : 443 (270–580)
Comparator : 470 (270–604)


## Reproducibility assertions
These checks make the notebook fail visibly if the frozen metadata no longer reproduce the revised manuscript denominators and statistics.


In [8]:
assert len(ann)==1018 and ann.patient_id.nunique()==1010
assert len(eligible)==883 and eligible.patient_id.nunique()==875
assert len(cohort)==500 and len(selected_patients)==498
assert len(remaining)==383 and len(remaining_patients)==382
assert len(overlap)==5
assert len(comparator)==378 and len(comparator_patients)==377
assert len(excluded)==5 and selected_patients.isdisjoint(comparator_patients)
assert len(negative)==135 and len(negative_patients)==135 and len(negative_overlap)==0
assert (counts==1).sum()==867 and (counts==2).sum()==8 and counts.max()==2
assert len(all_patients)==1010 and len(old_patients)==512
assert len(sel)==498 and len(cmp)==377
# Verify exact statistics produced from the frozen metadata
assert (ms, q1s, q3s) == (443.0, 270.0, 579.5)
assert (mc, q1c, q3c) == (470.0, 270.0, 604.0)

# Verify inferential result
assert U == 89461.0
assert round(p, 6) == 0.233373

# Verify manuscript reporting after rounding to whole DICOM files
assert f"{ms:.0f} ({q1s:.0f}–{q3s:.0f})" == "443 (270–580)"
assert f"{mc:.0f} ({q1c:.0f}–{q3c:.0f})" == "470 (270–604)"

print("ALL COHORT AND TABLE 1 AUDIT ASSERTIONS PASSED")


ALL COHORT AND TABLE 1 AUDIT ASSERTIONS PASSED


## Interpretation
The audit establishes an internally consistent sampling frame:

- **883 eligible scans from 875 patients**
- **500 selected scans from 498 patients**
- **378 comparison scans from 377 patients**
- **5 additional eligible scans** belong to patients already represented in the selected cohort and are excluded to preserve patient-disjoint groups.

For the corrected patient-level DICOM comparison, median file counts are **443 (IQR 270–580)** versus **470 (IQR 270–604)**, with **Mann–Whitney U = 89,461, p = 0.233373**.

The previous denominator of **512** is reproducible as **1,010 − 498**, i.e., all other patient directories in the broader download inventory. Because that population is not restricted to the eligible annotated sampling frame, it is not used in the corrected Table 1.

This audit does not alter the fixed 500-scan cohort, patient-level train/validation/test partitions, model training, segmentation results, figures, or conclusions.
